This is for creating balanced images for the training data set

In [1]:
import rasterio
from rasterio.windows import Window
from rasterio.mask import mask
import geopandas as gpd
import numpy as np
from shapely.geometry import box
import random

In [ ]:
def create_balanced_clips(
    image_path,
    water_polygons_path,
    output_folder,
    clip_size=1000,  # pixels
    num_clips=10
):
    """
    Create balanced clips containing roughly equal water/non-water pixels
    
    Parameters:
    image_path: Path to Sentinel-2 image
    water_polygons_path: Path to shapefile containing water bodies
    output_folder: Where to save clips
    clip_size: Size of square clips in pixels
    num_clips: Number of clips to create
    """
    # Read the water polygons
    water_gdf = gpd.read_file(water_polygons_path)
    
    with rasterio.open(image_path) as src:
        # Get image bounds
        bounds = src.bounds
        transform = src.transform
        crs = src.crs
        
        # Create a grid of possible clip locations
        x_steps = int((bounds.right - bounds.left) / (clip_size * transform[0]))
        y_steps = int((bounds.top - bounds.bottom) / (clip_size * transform[0]))
        
        clips = []
        for _ in range(num_clips):
            found_balanced_clip = False
            attempts = 0
            
            while not found_balanced_clip and attempts < 50:
                # Random starting point
                x_start = random.randint(0, x_steps - 1) * clip_size
                y_start = random.randint(0, y_steps - 1) * clip_size
                
                # Create window
                window = Window(x_start, y_start, clip_size, clip_size)
                
                # Get window bounds
                window_bounds = rasterio.windows.bounds(window, transform)
                clip_box = box(*window_bounds)
                
                # Calculate water percentage in clip
                clip_gdf = gpd.GeoDataFrame(geometry=[clip_box], crs=crs)
                intersection = gpd.overlay(clip_gdf, water_gdf, how='intersection')
                
                if intersection.empty:
                    water_percentage = 0
                else:
                    water_percentage = intersection.area.sum() / clip_box.area
                
                # Check if clip has roughly balanced water/non-water (30-70% water)
                if 0.3 <= water_percentage <= 0.7:
                    found_balanced_clip = True
                    clips.append({
                        'window': window,
                        'bounds': window_bounds,
                        'water_percentage': water_percentage
                    })
                
                attempts += 1
        
        # Save clips
        for i, clip in enumerate(clips):
            # Read data
            data = src.read(window=clip['window'])
            
            # Create output profile
            out_profile = src.profile.copy()
            out_profile.update({
                'height': clip_size,
                'width': clip_size,
                'transform': rasterio.windows.transform(clip['window'], transform)
            })
            
            # Save clip
            output_path = f"{output_folder}/clip_{i}.tif"
            with rasterio.open(output_path, 'w', **out_profile) as dst:
                dst.write(data)
            
            # Also save the clip bounds as a shapefile for reference
            clip_gdf = gpd.GeoDataFrame(
                {
                    'water_percentage': [clip['water_percentage']],
                    'geometry': [box(*clip['bounds'])]
                },
                crs=crs
            )
            clip_gdf.to_file(f"{output_folder}/clip_{i}_bounds.shp")

In [ ]:
def validate_clip_balance(clip_path, labels_path):
    """
    Check the actual water/non-water balance in a clip
    
    Parameters:
    clip_path: Path to image clip
    labels_path: Path to corresponding labels
    """
    with rasterio.open(labels_path) as src:
        labels = src.read(1)
        
    water_pixels = np.sum(labels == 1)
    non_water_pixels = np.sum(labels == 0)
    total_pixels = water_pixels + non_water_pixels
    
    water_percentage = (water_pixels / total_pixels) * 100
    
    print(f"Water pixels: {water_pixels} ({water_percentage:.1f}%)")
    print(f"Non-water pixels: {non_water_pixels} ({100-water_percentage:.1f}%)")